<a href="https://colab.research.google.com/github/miguelmccormickudg/Gravedad-Territorial-Relativista/blob/main/Copy_of_Untitled19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# -------------------------------
# Paso 1: Leer Masa Total Ajijic
# -------------------------------
masa_df = pd.read_csv("/content/masa_total_ajijic.csv")
masa_total = masa_df.loc[0, "masa_total"]
print(f"✅ Masa total de Ajijic cargada: {masa_total}")

# -------------------------------
# Paso 2: Leer matriz de distancias
# -------------------------------
df_dist = pd.read_csv("/content/distance_matrix.csv", index_col=0)
D_ij = df_dist.values

# Prevenir división por cero en la diagonal
D_ij = np.where(D_ij == 0, 1e-6, D_ij)

# -------------------------------
# Paso 3: Leer entropía por listing
# -------------------------------
df_entropy = pd.read_csv("/content/listings_with_entropy.csv")
df_entropy["node_id"] = df_entropy.index

# -------------------------------
# Paso 4: Leer estabilidad por host
# -------------------------------
df_stability = pd.read_csv("/content/E_i_resultado.csv")
df_stability.rename(columns={"E_i": "S_i"}, inplace=True)
# Add 'node_id' column to df_stability
df_stability["node_id"] = df_stability.index

# -------------------------------
# Paso 5: Preparar DataFrame maestro
# -------------------------------
# Asignar identificador de node_id si no existe
df_master = df_entropy[["node_id", "host_name", "entropy_weight"]].copy()

# Unir estabilidad por host_name
# Modified: Merge on 'node_id' instead of 'host_name'
df_master = pd.merge(
    df_master,
    df_stability[["node_id", "S_i"]], # Include 'node_id' in df_stability for merging
    on="node_id",
    how="left"
)

# Verificar consistencia
if df_master.isnull().any().any():
    print("⚠️ Advertencia: hay valores nulos tras la unión. Revisa consistencia de identificadores.")

# -------------------------------
# Paso 6: Extraer vectores
# -------------------------------
E_i = df_master["entropy_weight"].values
S_i = df_master["S_i"].values

n = len(E_i)
assert df_dist.shape[0] == n and df_dist.shape[1] == n, "La matriz de distancias no coincide con el número de nodos."

# -------------------------------
# Paso 7: Configurar exponente beta
# -------------------------------
beta = 1  # Cambia si lo necesitas

# -------------------------------
# Paso 8: Calcular F_ij
# -------------------------------
# El numerador es la misma constante para todos
numerador = masa_total

F_matrix = np.zeros((n, n))

# Calcular elemento por elemento
for i in range(n):
    for j in range(n):
        F_matrix[i, j] = (numerador / (D_ij[i, j] ** beta)) * E_i[i] * S_i[i]

# -------------------------------
# Paso 9: Guardar matriz final
# -------------------------------
output_path = "/content/F_ij_matrix.csv"
pd.DataFrame(F_matrix, index=df_master["node_id"], columns=df_master["node_id"]).to_csv(output_path)

print(f"\n✅ Matriz F_ij calculada y guardada en: {output_path}")

✅ Masa total de Ajijic cargada: 14135242.65
⚠️ Advertencia: hay valores nulos tras la unión. Revisa consistencia de identificadores.

✅ Matriz F_ij calculada y guardada en: /content/F_ij_matrix.csv
